<a href="https://colab.research.google.com/github/SoukouratouKarim/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [7]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "openai/gpt-oss-120b"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [10]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage

# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

answer, usage = ask_llm("In one sentence, what does a microfinance loan officer do?")
print(answer)
print(usage)

A microfinance loan officer evaluates, approves, and manages small‑scale loans for low‑income individuals or businesses, providing financial guidance and ensuring repayment compliance.
CompletionUsage(completion_tokens=53, prompt_tokens=93, total_tokens=146, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=13, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.124447186, prompt_time=0.013456459, completion_time=0.109479618, total_time=0.122936077)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [Double-click to edit]

### Part 1.2 — Temperature: the randomness dial

In [11]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

print("temperature = 0.0")
for i in range(5):
    answer, _ = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {answer}\n")

print("temperature = 1.2 ")
for i in range(5):
    answer, _ = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {answer}\n")

temperature = 0.0
1. **Name Ideas for a Savings Product Targeted at Market Traders in Accra**

| # | Suggested Name | Why It Works (Brief Rationale) |
|---|----------------|--------------------------------|
| 1 | **“Kokro‑Kokro Savings”** | *Kokro* means “small” in Twi – emphasizes that even tiny daily earnings can grow into a solid nest‑egg. |
| 2 | **“Bɔkɔɔ Bank”** | *Bɔkɔɔ* translates to “steady/consistent.” It signals a reliable, low‑risk way to keep money moving forward. |
| 3 | **“MarketMate Vault”** | Combines the familiar “market” vibe with a friendly “mate” – a partner that watches over traders’ cash. |
| 4 | **“Sika Sika Save”** | *Sika* = “money” in Twi; the repetition creates a rhythmic, memorable chant that traders can easily recall. |
| 5 | **“Adwuma Nest”** | *Adwuma* means “work” – a nest for the fruits of hard work, reinforcing the idea of saving the earnings from daily hustle. |
| 6 | **“Akwaaba Savings”** | *Akwaaba* = “welcome.” The product “welcomes” traders’ money

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

At temperature 0.0, outputs were highly repetitive but not perfectly identical the same handful of names (Kokro-Kokro Savings, Kente Kash) recurred across runs, showing the model is heavily biased toward high-probability tokens even without true determinism. At temperature 1.2, outputs varied widely in content, structure, and even format. For the loan decision-support system, low temperature (0 or close to it) is the right choice: extraction and summarization need to be factual, consistent, and reproducible not creative. A loan officer reviewing the same letter twice should get the same brief, not two different framings of an applicant's risk profile.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [12]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [39]:
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    prompt = f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    answer, _ = ask_llm(prompt, temperature=0)
    print(f" {letter_id} (V1) ")
    print(answer)
    print()

 L002 (V1) 
Kwame Boateng, a commercial driver in Kumasi, is requesting an urgent loan of GHS 25,000 to repair his trotro engine and cover personal debts. He notes that business is currently slow but expects improvement after the festive season, and he promises to repay the loan as soon as funds become available, though he has no collateral at this time.

 L006 (V1) 
**Summary**

Kofi, a 22‑year‑old entrepreneur, is requesting a GHS 50,000 loan to launch three ventures—a car‑washing service, a provision shop, and an import business for phones from Dubai. He has not yet started any of these businesses but claims to be energetic, business‑oriented, and trustworthy (though he has no collateral). He proposes to repay the loan within one year once the businesses become profitable.



In [32]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

SUMMARY_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
Your job is to summarize loan application letters into short, factual briefs.
Rules:
- Be strictly factual and neutral. Do not add opinions, judgments, or recommendations.
- Do not invent or infer any detail that is not explicitly stated in the letter.
- Write exactly 3-4 sentences.
- Use plain, scannable language a busy loan officer can read in seconds."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id])
    answer, _ = ask_llm(prompt, system_prompt=SUMMARY_SYSTEM_PROMPT, temperature=0)
    print(f" {letter_id} (V2) ")
    print(answer)
    print()

 L002 (V2) 
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He says the amount is needed urgently to repair his trotro engine and to settle personal debts. He reports that business has been slow but expects it to improve after the festive season and plans to repay the loan when funds become available. He currently has no collateral to offer.

 L006 (V2) 
Kofi, a 22‑year‑old applicant, requests a loan of GHS 50,000 to start a car‑washing business, a provision shop, and to import phones from Dubai. He has not yet begun any of these enterprises and offers no collateral for the loan. He proposes to repay the full amount within one year, contingent on the businesses becoming profitable.



**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** V1's naive prompt produced summaries that read cleanly but blurred the line between fact and applicant claim, and sometimes buried material risk information. For L006, V1 uncritically repeated Kofi's self-description ("claims to be energetic, business-minded, and trustworthy") while omitting that none of his three proposed businesses have actually started arguably the single most important fact for a loan officer. V2's system prompt, which explicitly instructed "no invented details" and factual neutrality, fixed this: it dropped the unverifiable self-praise and surfaced the missing-track-record detail, and consistently hedged applicant statements with "he says/reports/proposes" rather than stating them as settled fact.

The "no invented details" instruction matters because LLMs are prone to hallucination confidently generating plausible-sounding but unstated or fabricated information. In a decision-support system, a hallucinated detail (an invented collateral amount, an assumed repayment history) could directly mislead a loan officer's judgment.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [33]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance institution.
You extract structured fields from loan application letters.
Return ONLY a JSON object with EXACTLY these keys, no other text, no markdown fences:
- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

Rules:
- If a field is not explicitly stated in the letter, use null. Do not guess or infer.
- amount_ghs and repayment_months must be numbers only (no currency symbols or units).
- has_collateral_or_guarantor is true only if the letter mentions collateral, a guarantor, or a pledged asset."""

FEWSHOT_LETTER = """Dear Sir,
My name is Ama Serwaa, I sell fabric at Kejetia Market. I would like GHS 6,000 to restock
ahead of the Easter season. Business is unpredictable so I cannot say my exact profit.
I will pay back within 10 months. My brother, a mechanic, will guarantee this loan."""

FEWSHOT_OUTPUT = """{"applicant_name": "Ama Serwaa", "amount_ghs": 6000, "purpose": "restock fabric ahead of Easter season", "monthly_profit_ghs": null, "has_collateral_or_guarantor": true, "repayment_months": 10}"""

EXTRACT_PROMPT = """Here is an example.

Letter:
{fewshot_letter}

Output:
{fewshot_output}

Now extract the fields from this letter. Return ONLY the JSON object.

Letter:
{letter_text}

Output:"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

import json

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(
        fewshot_letter=FEWSHOT_LETTER,
        fewshot_output=FEWSHOT_OUTPUT,
        letter_text=letter_text,
    )
    raw, _ = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=0, max_tokens=1000)
    # Strip ```json fences if present
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"Warning: failed to parse JSON. Raw output:\n{raw}")
        return None

NO_NULL_SYSTEM_PROMPT = EXTRACT_SYSTEM_PROMPT.replace(
    "If a field is not explicitly stated in the letter, use null. Do not guess or infer.",
    "Fill in your best estimate for every field."
)
raw, _ = ask_llm(EXTRACT_PROMPT.format(fewshot_letter=FEWSHOT_LETTER, fewshot_output=FEWSHOT_OUTPUT, letter_text=LETTERS["L002"]), system_prompt=NO_NULL_SYSTEM_PROMPT, temperature=0, max_tokens=1000)
print(raw)

print(GOLD["L001"])
print(GOLD["L003"])
print(GOLD["L006"])
# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

import pandas as pd

results = {}
for letter_id, letter_text in LETTERS.items():
    results[letter_id] = extract_fields(letter_text)

df = pd.DataFrame(results).T
df



{"applicant_name":"Kwame Boateng","amount_ghs":25000,"purpose":"repair trotro engine and settle personal debts","monthly_profit_ghs":null,"has_collateral_or_guarantor":false,"repayment_months":null}
{'applicant_name': 'Akosua Mensah', 'amount_ghs': 8000, 'purpose': 'buy deep freezer / expand into frozen foods', 'monthly_profit_ghs': 900, 'has_collateral_or_guarantor': True, 'repayment_months': 20}
{'applicant_name': 'Efua Darko', 'amount_ghs': 15000, 'purpose': 'industrial sewing machines and fabric stock', 'monthly_profit_ghs': 2800, 'has_collateral_or_guarantor': True, 'repayment_months': 15}
{'applicant_name': 'Kofi', 'amount_ghs': 50000, 'purpose': 'car wash, provision shop, phone imports', 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': 12}


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900,True,20
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,None,False,None
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800,True,15
L004,Yaw Owusu,12000,feed and 500 new layers,1500,True,18
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,None,True,16
L006,Kofi,50000,"start a car washing business, a provision shop...",None,False,12


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** Checked against gold labels, extraction was accurate for L001 and L003 (all fields matched). L006 revealed a reliability issue: across separate runs with identical inputs and temperature=0, repayment_months was extracted correctly as 12 in one run but returned as null in another confirming that temperature=0 reduces but does not eliminate run-to-run variation. Testing without the "use null, don't guess" instruction on L006 (the letter with the most missing information) showed the model still correctly returned null rather than fabricating values, suggesting gpt-oss-120b's instruction-tuning already discourages guessing on structured extraction the explicit null rule reinforces this behavior rather than being the only safeguard against it.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [34]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.


BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
Given a loan application letter, produce a decision-support brief with three sections:

Strengths: factual points in the applicant's favor (e.g. stated profit, collateral, clear purpose)
Risks: factual concerns (e.g. no collateral, vague repayment plan, no track record, high amount relative to profit)
Suggested Next Step: one concrete action for the loan officer (e.g. "request proof of income", "verify guarantor identity", "request smaller loan amount")

Rules:
- You are NOT approving or rejecting the loan. Never write "approve," "reject," "deny," or state a final decision.
- Base every point strictly on what is stated in the letter. Do not invent details.
- Keep each section to 2-3 bullet points.
- Be neutral and factual, not judgmental about the applicant."""

BRIEF_PROMPT = "Loan application letter:\n\n{letter_text}\n\nProduce the decision-support brief."

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

def generate_brief(letter_text):
    prompt = BRIEF_PROMPT.format(letter_text=letter_text)
    answer, _ = ask_llm(prompt, system_prompt=BRIEF_SYSTEM_PROMPT, temperature=0, max_tokens=500)
    return answer

for letter_id in ["L002", "L006"]:
    print(f" {letter_id} ")
    print(generate_brief(LETTERS[letter_id]))
    print()

 L002 
**Strengths**
- Clearly states the purpose of the loan: engine repair and settlement of personal debts.  
- Identifies a specific occupation (commercial driver) that provides a regular source of income.  
- Indicates a seasonal expectation that business will improve after the festive period.  

**Risks**
- No collateral is offered to secure the loan.  
- Repayment plan is vague (“pay back whenever the money comes”) with no timeline or amount specified.  
- Business is described as “slow,” and no financial figures (e.g., earnings or profit) are provided to gauge repayment capacity.  

**Suggested Next Step**
- Request documented proof of income (e.g., recent pay slips, bank statements, or earnings logs) and a more detailed repayment schedule.

 L006 
**Strengths**  
- Clearly states the loan amount needed (GHS 50,000) and the intended uses (car‑wash, provision shop, phone import).  
- Demonstrates motivation and confidence, noting personal energy and a reputation among friends fo

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [35]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
           "has_collateral_or_guarantor", "repayment_months"]
gold_letter_ids = ["L001", "L003", "L006"]

def fields_match(field, extracted_val, gold_val):
    if field in ("purpose", "applicant_name") and isinstance(extracted_val, str) and isinstance(gold_val, str):
        e, g = extracted_val.strip().lower(), gold_val.strip().lower()
        return e in g or g in e
    return extracted_val == gold_val

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

table_rows = []
for field in fields:
    row = {"field": field}
    correct_count = 0
    for letter_id in gold_letter_ids:
        extracted_val = results[letter_id].get(field)
        gold_val = GOLD[letter_id].get(field)
        is_match = fields_match(field, extracted_val, gold_val)
        row[letter_id] = "✓" if is_match else f"✗ ({extracted_val!r} vs {gold_val!r})"
        correct_count += is_match
    row["accuracy"] = f"{correct_count}/{len(gold_letter_ids)}"
    table_rows.append(row)

accuracy_df = pd.DataFrame(table_rows).set_index("field")
accuracy_df

,L001,L003,L006,accuracy
field,,,,
applicant_name,✓,✓,✓,3/3
amount_ghs,✓,✓,✓,3/3
purpose,✗ ('buy a deep freezer and expand into frozen ...,✓,"✗ ('start a car washing business, a provision ...",1/3
monthly_profit_ghs,✓,✓,✓,3/3
has_collateral_or_guarantor,✓,✓,✓,3/3
repayment_months,✓,✓,✓,3/3


### Part 4.2 — Reliability: is the system consistent?

In [36]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.


N_RUNS = 5
letter_id = "L006"
reliability_results = []

for i in range(N_RUNS):
    result = extract_fields(LETTERS[letter_id])
    reliability_results.append(result)
    print(f"Run {i+1}: {result}")

reliability_df = pd.DataFrame(reliability_results)
reliability_df

Run 1: {'applicant_name': 'Kofi', 'amount_ghs': 50000, 'purpose': 'start a car washing business, a provision shop, and import phones from Dubai', 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': 12}
Run 2: {'applicant_name': 'Kofi', 'amount_ghs': 50000, 'purpose': 'start a car washing business, a provision shop, and import phones from Dubai', 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': None}
Run 3: {'applicant_name': 'Kofi', 'amount_ghs': 50000, 'purpose': 'start a car washing business, a provision shop, and import phones from Dubai', 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': 12}
Run 4: {'applicant_name': 'Kofi', 'amount_ghs': 50000, 'purpose': 'start a car washing business, a provision shop, and import phones from Dubai', 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': 12}
Run 5: {'applicant_name': 'Kofi', 'amount_ghs': 50000,

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,Kofi,50000,"start a car washing business, a provision shop...",None,False,12.0
1,Kofi,50000,"start a car washing business, a provision shop...",None,False,NaN
2,Kofi,50000,"start a car washing business, a provision shop...",None,False,12.0
3,Kofi,50000,"start a car washing business, a provision shop...",None,False,12.0
4,Kofi,50000,"start a car washing business, a provision shop...",None,False,12.0


### Part 4.3 — Hallucination probing

In [37]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
test1_question = f"""Based on this loan application letter, what is the applicant's credit score?

Letter:
{LETTERS['L002']}"""

test1_answer, _ = ask_llm(
    test1_question,
    system_prompt=SUMMARY_SYSTEM_PROMPT,
    temperature=0
)
print("TEST 1: Asking for a detail not in the letter (credit score) ")
print(test1_answer)
print()

#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
weather_report = """Today's weather in Accra: partly cloudy with a high of 31°C and a low of 24°C.
Humidity is around 78% with light winds from the southwest at 12 km/h. Chance of rain
this evening is 20%. Tomorrow will be sunnier with temperatures reaching 33°C."""

test2_result = extract_fields(weather_report)
print("TEST 2: Extracting fields from an irrelevant weather report")
print(test2_result)
# TODO: Record the outputs verbatim below and label each PASS or FAIL.

TEST 1: Asking for a detail not in the letter (credit score) 
The loan application does not mention a credit score for the applicant. No numerical credit rating or credit‑score information is provided in the letter. Therefore, the applicant’s credit score cannot be determined from the supplied text.

TEST 2: Extracting fields from an irrelevant weather report
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': None}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** Extraction accuracy: 5/6 fields scored 3/3 exact matches (name, amount, profit, collateral, repayment months). purpose scored 1/3 on strict string matching, but all three were actually correct in meaning just paraphrased differently from gold. It was hardest to score, not hardest to extract, since free-text fields need fuzzy matching instead of exact equality.

Reliability: 5 runs at temperature=0 on the same letter gave identical results on 5/6 fields, but repayment_months flipped to null once (4/5 correct). So temperature=0 reduces but doesn't guarantee determinism fields needing light inference (converting "within one year" → 12) are less stable than fields stated directly. Production systems shouldn't rely on temperature=0 alone; re-querying or flagging low-confidence fields for human review would help.

Hallucination probing: No hallucinations. Asked for a credit score not in the letter, the model said it wasn't provided. Given a weather report, the extractor returned null for every field instead of inventing an applicant. Minor gap: has_collateral_or_guarantor returned False instead of null (schema issue, not a hallucination the field isn't nullable). Fix: make all fields nullable, and add a pre-check step confirming the input is actually a loan letter before extracting.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**

1. Who could be unfairly harmed by full automation?

The system evaluates written English quality as a proxy for creditworthiness, even though it's never told to. L002 and L006 read as risky partly because the letters are short, informal, and vague ("pay back whenever the money comes," "when the businesses are booming") but vague phrasing is a writing-skill issue, not necessarily a business-viability issue. An applicant running a genuinely solid, cash-generating business a market trader who's profitable but has limited formal schooling, or someone who dictated their letter to a friend and it lost nuance in translation could be scored as high-risk purely because they can't perform "confident, detailed business letter" as well as a more educated or English-fluent applicant. This would systematically disadvantage exactly the population microfinance is supposed to serve: informal-sector traders, rural applicants, and non-native English speakers, while implicitly rewarding polish and literacy over actual repayment capacity. Full automation would encode this bias invisibly, since nothing in the pipeline currently checks for it.

2. Data privacy and cross-border implications

Loan letters contain names, financial details, occupations, and sometimes family/guarantor information personal data under Ghana's Data Protection Act. Sending it to Groq's API means that data leaves Ghana and is processed on infrastructure in another jurisdiction, subject to that country's laws rather than Ghana's, and Karim (or the institution) has no direct control over how it's retained, logged, or used downstream by the provider. Before deploying this at a real MFI, I'd check: (1) Groq's data retention and training-use policy does it store or use inputs to improve its models, and can that be disabled; (2) whether this setup complies with the Data Protection Act's rules on cross-border transfer of personal data (registration with the Data Protection Commission, applicant consent); (3) whether applicants are informed and consent to their letter being processed by a third-party AI system at all; (4) a contractual data processing agreement with the provider, if available at the tier being used.

3. Two concrete production safeguards

Mandatory human review before any decision communicated to an applicant. The system should never be allowed to output an approval or denial directly to a customer every brief goes to a loan officer who makes the actual call, exactly as the "no auto approve/reject" prompt rule enforces at the model level, but backed by a hard system-level rule too (no automated messaging pipeline connected to loan outcomes).
An appeal/override log with reasoning trail. Every generated brief, the officer's final decision, and any case where the officer disagreed with the AI's risk framing should be logged. This creates an audit trail to detect systematic bias (e.g., if letters from a particular language background or writing style are consistently flagged as riskier) and gives applicants a route to contest a decision that was influenced by the tool.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**

1. Prompting vs. hyperparameter tuning

Both are iterative, empirical processes: you change one thing, run it, look at the output, and adjust based on what you see there's no closed-form solution, just trial and observation, the same loop as tuning learning rate or batch size in Lab 3. The difference is what you're steering. Hyperparameters shape how a model learns from data over many training epochs; the model's actual knowledge is fixed once training starts. Prompting shapes what a frozen, pre-trained model does at inference time you're not changing any weights, just steering behavior through instructions, examples, and constraints. That makes prompt iteration much faster (seconds, not hours) but also less reliable in a specific sense: a hyperparameter change produces a new deterministic model, while the same prompt can still behave slightly differently run to run, as our temperature-0 reliability test showed.

2. Trust would I run this unattended?

No. The single result that most influenced this is the repayment_months field flipping between 12 and null across identical runs at temperature=0 even with zero randomness setting, the system wasn't 100% consistent on a field that directly affects how risky an applicant looks. If that instability exists on a field I explicitly tested, it likely exists elsewhere too, on inputs I haven't tested. Combined with the appropriateness concerns in 4.4 the risk of penalizing applicants for weak written English rather than weak business fundamentals this system is useful as a decision-support tool a human reviews, but not safe to run unattended for anything that affects whether someone gets a loan.

3. Cost and scale estimate

From the response.usage output we saw earlier (a simple test question used ~146 total tokens), and accounting for the fact that the real pipeline runs three calls per application summarization, extraction (with a few-shot example baked into every call), and brief generation, each with a system prompt and the full letter text a realistic per-application estimate is roughly 800–1,500 tokens total across all three calls, including the hidden reasoning tokens gpt-oss-120b uses. At 1,000 applications/month, that's roughly 1–1.5 million tokens/month. That volume comfortably fits within Groq's free tier for a pilot, but at real production scale (especially with retries for reliability, as suggested in 4.2) a paid/developer tier with higher rate limits becomes necessary provider choice should weigh not just per-token cost but rate limits, uptime guarantees, and data-handling terms (per the privacy concerns in 4.4), not just raw price.

4. API vs. training your own model

For a task like this, calling an API wins because you get a model that already has broad language understanding, reasoning, and world knowledge for free no need to collect thousands of labeled loan letters or spend days training and tuning a network like in Lab 3. Development time here was hours, not weeks, and the model handles messy, varied real-world letter text far better than anything I could train from scratch on six examples. Training your own model would make sense instead when: you have a large labeled dataset and a narrow, well-defined task where a small specialized model can match or beat a general one at a fraction of the inference cost; you need guaranteed low latency or offline/on-device operation; or you need full control over the model for privacy reasons (avoiding sending sensitive data to a third party, directly relevant to the cross-border data concern in 4.4).

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.